# Treino do modelo IC (LLaMA 2 + LoRA) no Google Colab

Este notebook treina o modelo de metadados ambientais usando a GPU gratuita do Colab.
Na RTX 4050 local levava ~60h; aqui costuma levar **2 a 4 horas**.

**Antes de começar:**
1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU** (T4 serve).
2. Coloque a pasta do projeto `projeto-ic` no seu Google Drive, em `MyDrive/projeto-ic`.
   A pasta precisa conter `src/`, `configs/`, `data/raw_json/` (com seus 663 JSONs) e `benchmarks/`.
3. Rode as células na ordem.

> Dica: se o Colab desconectar no meio do treino, é só reabrir, rodar as células 1 a 5 de novo
> e usar a **célula de RETOMAR** (os checkpoints ficam salvos no seu Drive).

## 1. Verificar a GPU

In [ ]:
!nvidia-smi

## 2. Conectar o Google Drive
Vai abrir um pop-up pedindo autorização da sua conta Google.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Entrar na pasta do projeto
Ajuste o caminho abaixo se você colocou o projeto em outro lugar do Drive.

In [ ]:
import os
PROJETO = "/content/drive/MyDrive/projeto-ic"
os.chdir(PROJETO)
print("Pasta atual:", os.getcwd())
print("Conteúdo:", os.listdir())
# Confere se os JSONs estão lá
n_json = len([f for f in os.listdir("data/raw_json") if f.endswith(".json")])
print(f"JSONs em data/raw_json: {n_json}")

## 4. Instalar as bibliotecas
O Colab já vem com PyTorch e CUDA. Instalamos o resto. Leva 1-2 minutos.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets sentencepiece sacrebleu rouge-score pyyaml
print("\nBibliotecas instaladas.")
import torch
print("CUDA disponível:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma")

## 5. Fazer login no Hugging Face
O LLaMA 2 exige aceitar a licença e autenticar. Cole seu token quando pedir.
- Aceite a licença: https://huggingface.co/meta-llama/Llama-2-7b-chat-hf
- Pegue um token (tipo *Read*): https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 6. Gerar o dataset (com o glossário e os negativos)
Roda as 4 etapas: estatísticas → glossário → gerador de perguntas → adiciona negativos.
Se você já gerou o `dataset_v3.jsonl` no PC e o subiu para o Drive, pode **pular esta célula**.

In [ ]:
!python -m src.analytics.compute --in data/raw_json --out data/analytics/stats.json
!python -m src.glossary.build --json-dir data/raw_json --out data/glossary/glossary_full.json
!python -m src.prompts.generator --config configs/dataset_v3.yaml
!python -m src.prompts.add_negatives --dataset data/processed/dataset_v3.jsonl --json-dir data/raw_json --n 2500

# Confere o tamanho e a mistura final
import json
from collections import Counter
c = Counter()
for l in open("data/processed/dataset_v3.jsonl", encoding="utf-8"):
    c[json.loads(l)["meta"]["kind"].split("_")[0]] += 1
print("Total:", sum(c.values()), "| mistura:", dict(c))

## 7. Treinar
Os checkpoints são salvos no seu Drive a cada 500 passos, então nada se perde se cair.
Vai aparecer o progresso com ETA. Numa T4, espere ~2-4h para as 3 épocas.

In [ ]:
!python -m src.training.train --config configs/exp01_colab.yaml

## 7b. RETOMAR (só se o Colab caiu no meio do treino)
Rode as células 1 a 5 de novo, depois esta. Ela continua do último checkpoint salvo no Drive.

In [ ]:
!python -m src.training.train --config configs/exp01_colab.yaml --resume

## 8. Avaliar
Roda o benchmark e mostra as métricas (F1, BLEU, ROUGE-L, acurácia de recusa).

In [ ]:
!python -m src.eval.run_eval --config configs/exp01_colab.yaml --benchmark benchmarks/gold_qa.jsonl

## 9. Conversar com o modelo (teste rápido)
Faz algumas perguntas direto pelo notebook para você ver a qualidade.

In [ ]:
import torch, yaml
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from src.common.prompts import build_chat_prompt

cfg = yaml.safe_load(open("configs/exp01_colab.yaml"))
m = cfg["model"]
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
base = AutoModelForCausalLM.from_pretrained(m["base"], quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(base, m["output_dir"]); model.eval()
tok = AutoTokenizer.from_pretrained(m["output_dir"])
if tok.pad_token_id is None: tok.pad_token = tok.eos_token
sistema = cfg["system_prompt"]

def perguntar(q):
    prompt = build_chat_prompt(sistema, [], new_user_message=q)
    ins = tok(prompt, return_tensors="pt", truncation=True, max_length=1536).to(model.device)
    with torch.inference_mode():
        out = model.generate(**ins, max_new_tokens=256, do_sample=False,
                             repetition_penalty=1.05, pad_token_id=tok.eos_token_id)
    print("P:", q)
    print("R:", tok.decode(out[0, ins["input_ids"].shape[-1]:], skip_special_tokens=True).strip())
    print()

for q in ["O que é mean_doppler_velocity?",
          "Para que serve reflectivity?",
          "Qual a cotação do dólar hoje?",
          "Preciso de dados de umidade em Southern Great Plains. O que uso?"]:
    perguntar(q)

## 10. (Opcional) Baixar o modelo treinado para o PC
O adapter LoRA já está no seu Drive (`checkpoints/exp01`). Se quiser um .zip para baixar:

In [ ]:
import shutil
shutil.make_archive("/content/adapter_exp01", "zip",
                    "/content/drive/MyDrive/projeto-ic/checkpoints/exp01")
from google.colab import files
files.download("/content/adapter_exp01.zip")